# Run Log Diagnosis

Paste a **run ID** (from `run_logs/`) or a **backtest log path** (from `backtests/`) in the cell below.

This notebook answers: **where did the strategy lose money and why?**

Sections:
1. PnL overview & drawdown
2. Trade-by-trade edge analysis
3. Position limits & capacity waste
4. Adverse selection (bought before drops / sold before rallies)
5. Fair-value tracking quality (for EMA-based products)

In [ ]:
# ── CONFIGURE: set ONE of these ──
RUN_ID = 98277                  # numeric ID from run_logs/
# BACKTEST_LOG = "backtests/2026-04-15_18-24-28.log"  # or a backtest .log path

In [ ]:
import json, os, re, sys
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (14, 4)})

# ── Load run data ──────────────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if "shared" in os.getcwd() else os.getcwd()

def load_run_log(run_id):
    """Parse a run_logs/{id}/ directory."""
    d = os.path.join(PROJECT_ROOT, "run_logs", str(run_id))
    with open(os.path.join(d, f"{run_id}.json")) as f:
        summary = json.load(f)
    with open(os.path.join(d, f"{run_id}.log")) as f:
        detail = json.load(f)
    return summary, detail

def load_backtest_log(path):
    """Parse a prosperity4bt backtest .log file."""
    if not os.path.isabs(path):
        path = os.path.join(PROJECT_ROOT, path)
    with open(path) as f:
        text = f.read()
    # Split on "Trade History:" marker
    parts = text.split("Trade History:")
    sandbox_text = parts[0].replace("Sandbox logs:", "").strip()
    trade_text = parts[1].strip() if len(parts) > 1 else "[]"
    # Parse trades (prosperity4bt uses trailing commas, use a lenient parser)
    trade_text = re.sub(r",\s*([}\]])", r"\1", trade_text)
    trades = json.loads(trade_text)
    return None, {"tradeHistory": trades, "activitiesLog": ""}

# Choose source
if "BACKTEST_LOG" in dir() and BACKTEST_LOG:  # noqa: F821
    summary, detail = load_backtest_log(BACKTEST_LOG)  # noqa: F821
    source_label = BACKTEST_LOG  # noqa: F821
else:
    summary, detail = load_run_log(RUN_ID)
    source_label = f"Run {RUN_ID}"

# ── Extract common structures ──
all_trades = detail.get("tradeHistory", [])
our_trades = [t for t in all_trades if t.get("buyer") == "SUBMISSION" or t.get("seller") == "SUBMISSION"]
bot_trades = [t for t in all_trades if t.get("buyer") != "SUBMISSION" and t.get("seller") != "SUBMISSION"]
products = sorted(set(t["symbol"] for t in our_trades))

# Parse PnL series from activitiesLog (run_logs only)
pnl_series = defaultdict(lambda: ([], []))  # product -> (timestamps, pnls)
act_log = detail.get("activitiesLog", "") or (summary.get("activitiesLog", "") if summary else "")
if act_log:
    for line in act_log.strip().split("\n")[1:]:
        parts = line.split(";")
        if len(parts) >= 17:
            ts, product, pnl = int(parts[1]), parts[2], float(parts[-1])
            pnl_series[product][0].append(ts)
            pnl_series[product][1].append(pnl)

# Graph PnL (run_logs only)
graph_ts, graph_pnl = [], []
if summary and "graphLog" in summary:
    for line in summary["graphLog"].strip().split("\n")[1:]:
        ts_str, val_str = line.split(";")
        graph_ts.append(int(ts_str))
        graph_pnl.append(float(val_str))

print(f"Source: {source_label}")
print(f"Products: {products}")
print(f"Total trades: {len(all_trades)} ({len(our_trades)} ours, {len(bot_trades)} bot-only)")
if summary:
    print(f"Final profit: {summary.get('profit', 'N/A')}")

## 1. PnL Overview & Drawdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Panel 1: PnL over time ──
ax = axes[0]
if graph_ts:
    ax.plot(graph_ts, graph_pnl, color="black", lw=1.5, label="Combined")
    ax.fill_between(graph_ts, graph_pnl, alpha=0.1, color="black")
for product in products:
    ts_list, pnl_list = pnl_series[product]
    if ts_list:
        ax.plot(ts_list, pnl_list, lw=1, alpha=0.7, label=product)
ax.axhline(0, color="gray", lw=0.5)
ax.set_title("PnL Over Time")
ax.set_xlabel("Timestamp")
ax.set_ylabel("PnL")
ax.legend(fontsize=8)

# Annotate max drawdown
if graph_pnl:
    peak = graph_pnl[0]
    max_dd, max_dd_ts = 0, 0
    for t, v in zip(graph_ts, graph_pnl):
        peak = max(peak, v)
        dd = v - peak
        if dd < max_dd:
            max_dd = dd
            max_dd_ts = t
    ax.annotate(f"Max DD: {max_dd:.0f}", xy=(max_dd_ts, graph_pnl[graph_ts.index(max_dd_ts)]),
                xytext=(max_dd_ts, graph_pnl[graph_ts.index(max_dd_ts)] + 50),
                arrowprops=dict(arrowstyle="->", color="red"), color="red", fontsize=9)

# ── Panel 2: Per-product PnL summary ──
ax = axes[1]
final_pnls = {}
for product in products:
    ts_list, pnl_list = pnl_series[product]
    if pnl_list:
        final_pnls[product] = pnl_list[-1]
    else:
        # Compute from trades if no activities log
        pos, pnl_val = 0, 0.0
        for t in sorted(our_trades, key=lambda x: x["timestamp"]):
            if t["symbol"] != product:
                continue
            if t["buyer"] == "SUBMISSION":
                pos += t["quantity"]; pnl_val -= t["price"] * t["quantity"]
            else:
                pos -= t["quantity"]; pnl_val += t["price"] * t["quantity"]
        final_pnls[product] = pnl_val  # unrealized not included

if final_pnls:
    colors = ["forestgreen" if v >= 0 else "indianred" for v in final_pnls.values()]
    bars = ax.bar(final_pnls.keys(), final_pnls.values(), color=colors)
    for bar, v in zip(bars, final_pnls.values()):
        ax.text(bar.get_x() + bar.get_width() / 2, v, f"{v:,.0f}",
                ha="center", va="bottom" if v >= 0 else "top", fontsize=10)
    ax.axhline(0, color="black", lw=0.5)
    ax.set_title("Final PnL by Product")
    ax.set_ylabel("PnL")

plt.tight_layout()
plt.show()

# Print drawdown stats
if graph_pnl:
    peak = graph_pnl[0]
    dd_start, dd_end = 0, 0
    cur_dd_start = 0
    max_dd = 0
    for i, v in enumerate(graph_pnl):
        if v >= peak:
            peak = v
            cur_dd_start = i
        dd = v - peak
        if dd < max_dd:
            max_dd = dd
            dd_start = cur_dd_start
            dd_end = i
    print(f"Max drawdown: {max_dd:.0f} (t={graph_ts[dd_start]} to t={graph_ts[dd_end]})")
    print(f"Drawdown duration: {graph_ts[dd_end] - graph_ts[dd_start]} ticks")

## 2. Trade-by-Trade Edge Analysis

For each trade, **edge** = how far the fill price is from fair value. Positive edge = good fill, negative = we got picked off.

- EMERALDS fair value: 10000 (fixed)
- TOMATOES fair value: mid-price at the time of the trade (from activities log)

In [ ]:
# Build mid-price lookup from activities log
mid_lookup = {}  # (product, timestamp) -> mid_price
if act_log:
    for line in act_log.strip().split("\n")[1:]:
        parts = line.split(";")
        if len(parts) >= 17:
            mid_lookup[(parts[2], int(parts[1]))] = float(parts[15])

def get_fair(product, timestamp):
    if product == "EMERALDS":
        return 10000.0
    key = (product, timestamp)
    if key in mid_lookup:
        return mid_lookup[key]
    # Nearest timestamp
    prod_ts = [ts for (p, ts) in mid_lookup if p == product]
    if not prod_ts:
        return None
    nearest = min(prod_ts, key=lambda x: abs(x - timestamp))
    return mid_lookup[(product, nearest)]

# Compute edge for every trade
for product in products:
    pt = sorted([t for t in our_trades if t["symbol"] == product], key=lambda t: t["timestamp"])
    if not pt:
        continue

    edges = []
    timestamps = []
    sizes = []
    sides = []

    for t in pt:
        ts = t["timestamp"]
        p = t["price"]
        q = t["quantity"]
        fair = get_fair(product, ts)
        if fair is None:
            continue

        is_buy = t["buyer"] == "SUBMISSION"
        # Edge: for a buy, edge = fair - price (bought below FV = positive)
        #        for a sell, edge = price - fair (sold above FV = positive)
        edge = (fair - p) if is_buy else (p - fair)
        edges.append(edge)
        timestamps.append(ts)
        sizes.append(q)
        sides.append("BUY" if is_buy else "SELL")

    edges = np.array(edges)
    sizes = np.array(sizes)
    vol_weighted_edge = np.sum(edges * sizes) / np.sum(sizes) if np.sum(sizes) > 0 else 0

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Panel 1: Edge per trade over time
    colors = ["forestgreen" if e >= 0 else "indianred" for e in edges]
    axes[0].scatter(timestamps, edges, c=colors, s=sizes * 6, alpha=0.6, edgecolors="black", lw=0.3)
    axes[0].axhline(0, color="black", lw=0.5)
    axes[0].set_title(f"{product}: Edge per Trade Over Time")
    axes[0].set_xlabel("Timestamp")
    axes[0].set_ylabel("Edge (positive = good)")

    # Panel 2: Edge histogram
    axes[1].hist(edges, bins=30, color="steelblue", edgecolor="white")
    axes[1].axvline(0, color="red", lw=1, ls="--")
    axes[1].axvline(vol_weighted_edge, color="black", lw=1.5, label=f"Vol-wtd avg: {vol_weighted_edge:.2f}")
    axes[1].set_title(f"{product}: Edge Distribution")
    axes[1].set_xlabel("Edge")
    axes[1].legend(fontsize=8)

    # Panel 3: Fill price distribution by side
    buy_prices = [t["price"] for t in pt if t["buyer"] == "SUBMISSION"]
    sell_prices = [t["price"] for t in pt if t["seller"] == "SUBMISSION"]
    all_prices = sorted(set(buy_prices + sell_prices))
    x = np.arange(len(all_prices))
    buy_counts = [buy_prices.count(p) for p in all_prices]
    sell_counts = [sell_prices.count(p) for p in all_prices]
    axes[2].bar(x - 0.15, buy_counts, 0.3, label="Buys", color="forestgreen", alpha=0.8)
    axes[2].bar(x + 0.15, sell_counts, 0.3, label="Sells", color="indianred", alpha=0.8)
    axes[2].set_xticks(x)
    axes[2].set_xticklabels([str(int(p)) for p in all_prices], rotation=45, fontsize=7)
    axes[2].set_title(f"{product}: Fill Price Distribution")
    axes[2].legend(fontsize=8)
    if product == "EMERALDS" and 10000.0 in all_prices:
        fv_idx = all_prices.index(10000.0)
        axes[2].axvline(fv_idx, color="red", ls="--", lw=1, alpha=0.5)

    plt.suptitle(f"{product}: vol-weighted edge = {vol_weighted_edge:.2f} | "
                 f"positive = {np.sum(edges > 0)}/{len(edges)} trades",
                 fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

    # Print zero-edge breakdown (mainly for EMERALDS)
    zero_edge = [e for e in edges if abs(e) < 0.01]
    if len(zero_edge) > 5:
        zero_vol = sum(s for e, s in zip(edges, sizes) if abs(e) < 0.01)
        total_vol = sum(sizes)
        print(f"  WARNING: {len(zero_edge)}/{len(edges)} trades ({zero_vol}/{total_vol} volume = "
              f"{zero_vol/total_vol*100:.0f}%) have zero edge")
    neg_edge = [(e, s) for e, s in zip(edges, sizes) if e < -0.5]
    if neg_edge:
        neg_vol = sum(s for _, s in neg_edge)
        neg_pnl = sum(e * s for e, s in neg_edge)
        print(f"  {len(neg_edge)} negative-edge trades: total impact = {neg_pnl:,.0f}")

## 3. Position & Capacity

Position over time, how often we hit the limit, and whether we're stuck unable to trade.

In [ ]:
POS_LIMIT = 20
n_products = len(products)
fig, axes = plt.subplots(n_products, 2, figsize=(15, 4 * n_products), squeeze=False)

for row_idx, product in enumerate(products):
    pt = sorted([t for t in our_trades if t["symbol"] == product], key=lambda t: t["timestamp"])
    if not pt:
        continue

    # Reconstruct position
    pos = 0
    positions, ts_list = [0], [0]
    at_limit_ts = []

    for t in pt:
        if t["buyer"] == "SUBMISSION":
            pos += t["quantity"]
        else:
            pos -= t["quantity"]
        positions.append(pos)
        ts_list.append(t["timestamp"])
        if abs(pos) >= POS_LIMIT:
            at_limit_ts.append(t["timestamp"])

    # Panel 1: Position over time
    ax = axes[row_idx][0]
    ax.fill_between(ts_list, positions, alpha=0.3, step="post",
                     color="steelblue" if product == "EMERALDS" else "coral")
    ax.step(ts_list, positions, where="post", lw=1.2,
            color="steelblue" if product == "EMERALDS" else "coral")
    ax.axhline(POS_LIMIT, color="red", ls=":", lw=0.8, label=f"+/- {POS_LIMIT} limit")
    ax.axhline(-POS_LIMIT, color="red", ls=":", lw=0.8)
    ax.axhline(0, color="black", lw=0.5)

    # Mark limit hits
    for lt in at_limit_ts:
        ax.axvline(lt, color="red", alpha=0.15, lw=2)
    ax.set_title(f"{product}: Position Over Time ({len(at_limit_ts)} limit hits)")
    ax.set_xlabel("Timestamp")
    ax.set_ylabel("Position")
    ax.legend(fontsize=8)

    # Panel 2: Time spent at each position level
    ax = axes[row_idx][1]
    # Weight by duration between trades
    pos_durations = defaultdict(int)
    for i in range(len(ts_list) - 1):
        duration = ts_list[i + 1] - ts_list[i]
        pos_durations[positions[i]] += duration
    total_time = sum(pos_durations.values())

    pos_levels = sorted(pos_durations.keys())
    durations = [pos_durations[p] for p in pos_levels]
    colors = ["red" if abs(p) >= POS_LIMIT else ("orange" if abs(p) >= POS_LIMIT - 2 else "steelblue")
              for p in pos_levels]
    ax.bar(pos_levels, durations, color=colors)
    ax.set_title(f"{product}: Time at Each Position Level")
    ax.set_xlabel("Position")
    ax.set_ylabel("Duration (ticks)")

    # Compute stats
    at_limit_time = sum(d for p, d in pos_durations.items() if abs(p) >= POS_LIMIT)
    near_limit_time = sum(d for p, d in pos_durations.items() if abs(p) >= POS_LIMIT - 2)

    if total_time > 0:
        ax.text(0.98, 0.95,
                f"At limit: {at_limit_time/total_time*100:.0f}%\n"
                f"Near limit: {near_limit_time/total_time*100:.0f}%",
                transform=ax.transAxes, va="top", ha="right", fontsize=9,
                bbox=dict(boxstyle="round", fc="lightyellow"))

plt.tight_layout()
plt.show()

## 4. Adverse Selection

Did we buy right before the price dropped, or sell right before it rallied? For each of our trades, we measure the **mark-to-market PnL 10 ticks later** to detect adverse selection.

In [ ]:
# Build sorted mid-price series per product for forward-looking analysis
mid_series = {}  # product -> sorted list of (timestamp, mid_price)
for (product, ts), mid in sorted(mid_lookup.items()):
    mid_series.setdefault(product, []).append((ts, mid))

LOOK_AHEAD = 10  # ticks to look forward (each tick = 100 units)

if mid_series:
    for product in products:
        pt = sorted([t for t in our_trades if t["symbol"] == product], key=lambda t: t["timestamp"])
        series = mid_series.get(product, [])
        if not pt or not series:
            continue

        ts_arr = np.array([s[0] for s in series])

        mtm_changes = []
        trade_ts = []
        trade_sides = []
        trade_sizes = []

        for t in pt:
            fill_ts = t["timestamp"]
            fill_price = t["price"]
            is_buy = t["buyer"] == "SUBMISSION"
            qty = t["quantity"]

            # Find mid price LOOK_AHEAD ticks later
            future_ts = fill_ts + LOOK_AHEAD * 100
            idx = np.searchsorted(ts_arr, future_ts)
            if idx >= len(series):
                idx = len(series) - 1

            future_mid = series[idx][1]

            # MTM change: how much did we make/lose from this fill
            if is_buy:
                mtm = (future_mid - fill_price) * qty
            else:
                mtm = (fill_price - future_mid) * qty

            mtm_changes.append(mtm)
            trade_ts.append(fill_ts)
            trade_sides.append("BUY" if is_buy else "SELL")
            trade_sizes.append(qty)

        mtm_changes = np.array(mtm_changes)

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))

        # Panel 1: MTM per trade
        colors = ["forestgreen" if m >= 0 else "indianred" for m in mtm_changes]
        axes[0].scatter(trade_ts, mtm_changes, c=colors,
                       s=[s * 8 for s in trade_sizes], alpha=0.6, edgecolors="black", lw=0.3)
        axes[0].axhline(0, color="black", lw=0.5)
        axes[0].set_title(f"{product}: MTM {LOOK_AHEAD} ticks after fill")
        axes[0].set_xlabel("Timestamp")
        axes[0].set_ylabel("MTM Change")

        # Panel 2: Cumulative adverse selection
        cum_mtm = np.cumsum(mtm_changes)
        axes[1].plot(range(len(cum_mtm)), cum_mtm,
                    color="steelblue" if product == "EMERALDS" else "coral")
        axes[1].axhline(0, color="black", lw=0.5)
        axes[1].set_title(f"{product}: Cumulative post-fill MTM")
        axes[1].set_xlabel("Trade #")
        axes[1].set_ylabel("Cumulative MTM")

        adverse = np.sum(mtm_changes < 0)
        total_adverse = np.sum(mtm_changes[mtm_changes < 0])
        total_favorable = np.sum(mtm_changes[mtm_changes >= 0])

        plt.suptitle(f"{product}: {adverse}/{len(mtm_changes)} trades adversely selected | "
                     f"Net post-fill MTM: {np.sum(mtm_changes):+,.0f} "
                     f"(adverse: {total_adverse:,.0f}, favorable: {total_favorable:,.0f})",
                     fontweight="bold", y=1.02, fontsize=10)
        plt.tight_layout()
        plt.show()
else:
    print("No activities log available -- adverse selection analysis requires mid-price data.")

## 5. Missed Opportunities

How much order flow arrived that we **didn't** capture? These are bot-only trades where our quotes could have filled.

In [ ]:
for product in products:
    our_ts = set(t["timestamp"] for t in our_trades if t["symbol"] == product)
    bot_only = [t for t in bot_trades if t["symbol"] == product]
    our_pt = [t for t in our_trades if t["symbol"] == product]

    our_vol = sum(t["quantity"] for t in our_pt)
    bot_vol = sum(t["quantity"] for t in bot_only)
    total_vol = our_vol + bot_vol

    # For each bot trade, check if it was at a price we would have wanted
    missed_at_our_prices = []
    for t in bot_only:
        fair = get_fair(product, t["timestamp"])
        if fair is None:
            continue
        price = t["price"]
        # Would have been a good buy (below FV) or good sell (above FV)?
        if price < fair - 0.5 or price > fair + 0.5:
            missed_at_our_prices.append(t)

    missed_vol = sum(t["quantity"] for t in missed_at_our_prices)

    fig, ax = plt.subplots(figsize=(12, 3))

    # Timeline: our trades vs bot trades
    our_ts_list = [t["timestamp"] for t in our_pt]
    our_prices = [t["price"] for t in our_pt]
    bot_ts_list = [t["timestamp"] for t in bot_only]
    bot_prices = [t["price"] for t in bot_only]

    ax.scatter(bot_ts_list, bot_prices, s=10, alpha=0.3, color="gray", label=f"Bot-only ({bot_vol} vol)")
    ax.scatter(our_ts_list, our_prices, s=30, alpha=0.7,
               color="steelblue" if product == "EMERALDS" else "coral",
               edgecolors="black", lw=0.3, label=f"Ours ({our_vol} vol)")

    # Highlight missed opportunities
    if missed_at_our_prices:
        m_ts = [t["timestamp"] for t in missed_at_our_prices]
        m_prices = [t["price"] for t in missed_at_our_prices]
        ax.scatter(m_ts, m_prices, s=40, alpha=0.5, color="gold", marker="x",
                   label=f"Missed ({missed_vol} vol)")

    if product == "EMERALDS":
        ax.axhline(10000, color="red", ls="--", lw=0.8, alpha=0.5)

    ax.set_title(f"{product}: Our Fills vs Market Activity")
    ax.set_xlabel("Timestamp")
    ax.set_ylabel("Price")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    capture_pct = our_vol / total_vol * 100 if total_vol > 0 else 0
    print(f"  {product}: captured {our_vol}/{total_vol} volume ({capture_pct:.0f}%) | "
          f"{len(missed_at_our_prices)} missed trades ({missed_vol} vol) had edge > 0.5")

## 6. Summary Scorecard

In [ ]:
print(f"{'='*65}")
print(f"  DIAGNOSIS SCORECARD: {source_label}")
print(f"{'='*65}")

for product in products:
    pt = sorted([t for t in our_trades if t["symbol"] == product], key=lambda t: t["timestamp"])
    if not pt:
        continue

    buys = [t for t in pt if t["buyer"] == "SUBMISSION"]
    sells = [t for t in pt if t["seller"] == "SUBMISSION"]
    buy_vol = sum(t["quantity"] for t in buys)
    sell_vol = sum(t["quantity"] for t in sells)
    buy_avg = sum(t["price"] * t["quantity"] for t in buys) / buy_vol if buy_vol else 0
    sell_avg = sum(t["price"] * t["quantity"] for t in sells) / sell_vol if sell_vol else 0
    net_pos = buy_vol - sell_vol

    # Reconstruct position for limit analysis
    pos = 0
    at_limit_count = 0
    for t in pt:
        if t["buyer"] == "SUBMISSION":
            pos += t["quantity"]
        else:
            pos -= t["quantity"]
        if abs(pos) >= POS_LIMIT:
            at_limit_count += 1

    # Edge stats
    edges = []
    for t in pt:
        fair = get_fair(product, t["timestamp"])
        if fair is None:
            continue
        is_buy = t["buyer"] == "SUBMISSION"
        edge = (fair - t["price"]) if is_buy else (t["price"] - fair)
        edges.append((edge, t["quantity"]))

    zero_vol = sum(q for e, q in edges if abs(e) < 0.01)
    neg_vol = sum(q for e, q in edges if e < -0.5)
    total_vol = buy_vol + sell_vol
    wtd_edge = sum(e * q for e, q in edges) / sum(q for _, q in edges) if edges else 0

    ts_range = pt[-1]["timestamp"] - pt[0]["timestamp"]

    print(f"\n  {product}:")
    print(f"    Trades:       {len(buys)} buys ({buy_vol} vol), {len(sells)} sells ({sell_vol} vol)")
    print(f"    Avg price:    buy {buy_avg:,.1f}, sell {sell_avg:,.1f}, spread {sell_avg - buy_avg:,.1f}")
    print(f"    Final pos:    {net_pos:+d}")
    print(f"    Wtd edge:     {wtd_edge:+.2f} per unit")
    print(f"    Limit hits:   {at_limit_count} times")

    # Flag problems
    problems = []
    if zero_vol / total_vol > 0.3 and total_vol > 0:
        problems.append(f"ZERO-EDGE: {zero_vol}/{total_vol} vol ({zero_vol/total_vol*100:.0f}%) at fair value")
    if at_limit_count > len(pt) * 0.2:
        problems.append(f"CAPACITY: hit position limit {at_limit_count}/{len(pt)} trades ({at_limit_count/len(pt)*100:.0f}%)")
    if abs(net_pos) > POS_LIMIT * 0.6:
        problems.append(f"INVENTORY: ending with {net_pos:+d} ({abs(net_pos)/POS_LIMIT*100:.0f}% of limit)")
    if wtd_edge < 0:
        problems.append(f"NEGATIVE EDGE: avg {wtd_edge:.2f} per unit -- getting picked off")
    if neg_vol / total_vol > 0.2 and total_vol > 0:
        problems.append(f"BAD FILLS: {neg_vol} vol ({neg_vol/total_vol*100:.0f}%) below fair value")

    if problems:
        print(f"    PROBLEMS:")
        for p in problems:
            print(f"      >> {p}")
    else:
        print(f"    No major issues detected.")

print(f"\n{'='*65}")